# Notebook 02 — Conversion Rate Test
**Skenario:** Apakah landing page baru meningkatkan conversion rate secara signifikan?  
**Tests:** Z-test for proportions + Chi-square + Confidence Interval

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency, norm
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from pathlib import Path

OUT     = Path('../output')
FIGURES = OUT / 'figures'

BLUE  = '#2563EB'
RED   = '#DC2626'
GREEN = '#059669'
GRAY  = '#6B7280'
ALPHA = 0.05

df = pd.read_parquet(OUT / 'df_clean.parquet')
ctrl  = df[df['group'] == 'control']
treat = df[df['group'] == 'treatment']

n_ctrl,  conv_ctrl  = len(ctrl),  ctrl['converted'].sum()
n_treat, conv_treat = len(treat), treat['converted'].sum()
p_ctrl  = conv_ctrl  / n_ctrl
p_treat = conv_treat / n_treat

print(f'Control   : n={n_ctrl:,}, conversions={conv_ctrl:,}, rate={p_ctrl:.4f}')
print(f'Treatment : n={n_treat:,}, conversions={conv_treat:,}, rate={p_treat:.4f}')
print(f'Absolute lift: {p_treat - p_ctrl:+.4f}')

## A. Hipotesis

- **H₀:** p_treatment = p_control (landing page baru tidak berbeda)
- **H₁:** p_treatment > p_control (landing page baru lebih baik)
- **α = 0.05** (significance level)
- **One-tailed test** (kita hanya peduli jika treatment lebih baik)

## B. Z-test untuk Proporsi

In [ ]:
count = np.array([conv_treat, conv_ctrl])
nobs  = np.array([n_treat,    n_ctrl])

z_stat, p_value_one = proportions_ztest(count, nobs, alternative='larger')
_, p_value_two      = proportions_ztest(count, nobs, alternative='two-sided')

print('=== Z-TEST UNTUK PROPORSI ===')
print(f'Z-statistic : {z_stat:.4f}')
print(f'p-value (one-tailed) : {p_value_one:.4f}')
print(f'p-value (two-tailed) : {p_value_two:.4f}')
print(f'\nKritical value (α=0.05, one-tailed): {norm.ppf(1 - ALPHA):.4f}')

if p_value_one < ALPHA:
    print(f'\n✅ TOLAK H₀ — perbedaan signifikan secara statistik (p={p_value_one:.4f} < {ALPHA})')
else:
    print(f'\n❌ GAGAL TOLAK H₀ — perbedaan TIDAK signifikan (p={p_value_one:.4f} ≥ {ALPHA})')

## C. Chi-square Test (Validasi)

In [ ]:
contingency = np.array([
    [conv_treat,          n_treat - conv_treat],
    [conv_ctrl,           n_ctrl  - conv_ctrl]
])

chi2, p_chi2, dof, expected = chi2_contingency(contingency)
cramers_v = np.sqrt(chi2 / (contingency.sum() * (min(contingency.shape) - 1)))

print('=== CHI-SQUARE TEST ===')
print(f'Chi2-statistic : {chi2:.4f}')
print(f'p-value        : {p_chi2:.4f}')
print(f'Degrees of freedom: {dof}')
print(f"Cramér's V (effect size): {cramers_v:.4f}")
print(f"  → Interpretasi: {'Kecil (<0.1)' if cramers_v < 0.1 else 'Sedang (0.1-0.3)' if cramers_v < 0.3 else 'Besar (>0.3)'}")

## D. Confidence Interval

In [ ]:
ci_ctrl  = proportion_confint(conv_ctrl,  n_ctrl,  alpha=ALPHA, method='wilson')
ci_treat = proportion_confint(conv_treat, n_treat, alpha=ALPHA, method='wilson')

# CI untuk perbedaan
diff      = p_treat - p_ctrl
se_diff   = np.sqrt(p_ctrl*(1-p_ctrl)/n_ctrl + p_treat*(1-p_treat)/n_treat)
z_crit    = norm.ppf(1 - ALPHA/2)
ci_diff   = (diff - z_crit * se_diff, diff + z_crit * se_diff)

print('=== CONFIDENCE INTERVALS (95%) ===')
print(f'Control   : [{ci_ctrl[0]:.4f}, {ci_ctrl[1]:.4f}]')
print(f'Treatment : [{ci_treat[0]:.4f}, {ci_treat[1]:.4f}]')
print(f'Difference: [{ci_diff[0]:+.4f}, {ci_diff[1]:+.4f}]')
print(f'  → CI difference mengandung 0? {ci_diff[0] < 0 < ci_diff[1]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart dengan error bar
groups   = ['Control\n(old page)', 'Treatment\n(new page)']
rates    = [p_ctrl,               p_treat]
errors   = [
    [p_ctrl  - ci_ctrl[0],  ci_ctrl[1]  - p_ctrl],
    [p_treat - ci_treat[0], ci_treat[1] - p_treat]
]
colors   = [BLUE, RED if p_treat < p_ctrl else GREEN]

bars = axes[0].bar(groups, [r*100 for r in rates], color=colors,
                   yerr=[[e[0]*100 for e in errors], [e[1]*100 for e in errors]],
                   capsize=8, error_kw={'linewidth': 2}, width=0.4)
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[0].set_title('Conversion Rate per Grup\n(bar = mean, whisker = 95% CI)', fontweight='bold')
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, rate*100 + 0.1,
                 f'{rate:.2%}', ha='center', fontweight='bold')

# CI difference plot
axes[1].axvline(0, color=GRAY, linestyle='--', linewidth=1.5, label='H₀: diff=0')
axes[1].errorbar([diff*100], [0], xerr=[[abs(ci_diff[0]-diff)*100], [abs(ci_diff[1]-diff)*100]],
                 fmt='o', color=BLUE, markersize=10, capsize=10, linewidth=2.5,
                 label=f'95% CI: [{ci_diff[0]*100:+.2f}%, {ci_diff[1]*100:+.2f}%]')
axes[1].set_xlabel('Perbedaan Conversion Rate (%)')
axes[1].set_yticks([])
axes[1].set_title('95% CI — Perbedaan Treatment vs Control', fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.savefig(FIGURES / 'B_conversion_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: B_conversion_test.png')

## E. Business Decision

In [ ]:
avg_order_value = df[df['converted'] == 1]['revenue'].mean()
daily_traffic   = df.groupby(df['timestamp'].dt.date)['user_id'].count().mean()

daily_conv_uplift    = (p_treat - p_ctrl) * daily_traffic
daily_revenue_uplift = daily_conv_uplift * avg_order_value
annual_uplift        = daily_revenue_uplift * 365

verdict = 'GO ✅' if p_value_one < ALPHA else 'NO-GO ❌'

print('=== BUSINESS DECISION ===')
print(f'Verdict        : {verdict}')
print(f'Alasan         : p-value={p_value_one:.4f} {"<" if p_value_one < ALPHA else "≥"} α={ALPHA}')
print(f'\nEstimasi impact jika diluncurkan:')
print(f'  Daily traffic       : {daily_traffic:,.0f} pengguna/hari')
print(f'  Conversion uplift   : {daily_conv_uplift:+.1f} konversi/hari')
print(f'  Revenue uplift/hari : USD {daily_revenue_uplift:+,.0f}')
print(f'  Revenue uplift/tahun: USD {annual_uplift:+,.0f}')
print(f'\nRekomendasi: {"Lanjutkan rollout ke 100% pengguna" if p_value_one < ALPHA else "Jangan diluncurkan — tidak ada bukti peningkatan signifikan"}')